<center>
    <img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="900">
</center>

# Yet another math for ML course

# Домашнее задание 3 (продвинутая группа): модификации градиентного спуска и ridge

В этой домашке две части. В первой вы реализуете логистическую регрессию и обучите её пятью разными версиями градиентного спуска — от обычного mini-batch до Adam — и посмотрите глазами, как они ведут себя на одной и той же задаче. 

Во второй выведете ridge-регрессию из условной оптимизации и увидите, что штраф за размер весов — это множитель Лагранжа.

Всего за домашку можно получить 10 баллов, плюс 0.5 за задание 0 и 1 за бонусное задание 9\*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

np.random.seed(42)

## Формат проверки работы: peer review

Каждая работа проверяется несколькими вашими случайно выбранными однокурсниками. Вы также обязаны проверить несколько чужих работ.

Проверяющий выставляет баллы по критериям из задания и оставляет содержательные комментарии: что сделано хорошо, где есть ошибки, что стоит исправить.

Итоговая оценка за задание складывается из трёх частей:

$$
\text{Оценка} = 0.8 \cdot \text{Решение} + 0.1 \cdot \text{Полезность ревью} + 0.1 \cdot \text{Согласованность ревью}
$$

- **80% решение** — это баллы за вашу собственную работу, полученные при проверке.
- **10% полезность ревью** — после проверки авторы работ оценивают, насколько полезными были полученные комментарии. Хорошее ревью должно быть конкретным, вежливым и помогать понять ошибки. Комментарии вида «всё ок», «неверно», «плохо» без объяснений считаются бесполезными.
- **10% согласованность ревью** — оценка проверяющего сравнивается с оценками других проверяющих. Если ваши баллы систематически сильно отличаются от общего консенсуса без убедительных комментариев, эта часть оценки снижается.

Максимум за домашку — 10 баллов, плюс 1.5 бонусных: 0.5 за задание 0 и 1 за задание 9\*. Итого можно набрать 11.5. Почти под каждым заданием есть вопросы. Ответы на них — часть задания и оцениваются наравне с кодом: работающий код с пустым «Ответ:» полного балла не получает.

Можно:

* указывать на ошибки и спорные места;
* снижать баллы, если решение не соответствует критериям;
* давать рекомендации по исправлению;
* писать развёрнутые комментарии по существу.

Нельзя:

* ставить баллы случайно или всем одинаково;
* писать формальные комментарии без содержания;
* оскорблять автора работы;
* договариваться о взаимных оценках;
* просить других поставить вам определённый балл;
* использовать peer review как способ отомстить или помочь друзьям.

Проверка тоже является частью задания. Халтурные, пустые или недобросовестные ревью будут снижать итоговый балл. Конкретные способы расчёта согласованности и полезности ревью не раскрываются.

__(0.5 балла, бонус) Задание 0: любимый мем__

**ФИО:**

**Ваш любимый мем.** Вставьте его сюда.

> **Для ревьюеров:** полный балл — если мем есть. Если не оч смешно баллы можно снять...

В первой части проверок почти нет — там всё видно на анимации. Во второй части после каждого шага стоит ячейка с проверкой: она не знает правильного ответа целиком, но знает, каким он обязан быть.

In [ ]:
def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

def numeric_derivative(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

# Часть 1: 50 оттенков градиентного спуска (6 баллов)

В этом задании вам предстоит реализовать линейный классификатор и натренировать его, используя различные модификации градинетного спуска. 

In [ ]:
from sklearn import datasets, preprocessing
from IPython import display

Для наших целей будем использовать искуственно сгенерированные данные.

In [ ]:
# keep random_state=42 for deterministic results
(X, y) = datasets.make_circles(n_samples=1024, shuffle=True, noise=0.2, factor=0.4, random_state=42)
ind = np.logical_or(y == 1, X[:, 1] > X[:, 0] - 0.5)
X = X[ind, :]
m = np.array([[1, 1], [-2, 1]])
X = preprocessing.scale(X)
y = y[ind]

plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=20)
plt.show()

__(0.5 балла) Задание 1.__ Как вы можете заметить, данные не являются линейно разделимыми. Нам придётся добавить в обучающую выборку новые фичи либо использовать нелинейные модели. Предположим, что разделяющая поверхность имеет вид окружности. Добавьте в матрицу признаков дополнительные колонки $x_1^2$, $x_2^2$ и $x_1 \cdot x_2$.

In [ ]:
def expand(X):
    """
    Добавляет квадратичные фичи. 
    Для каждой строки матрицы находит строку 
    [feature0, feature1, feature0^2, feature1^2, feature0*feature1, 1]
    
    :param X: матрица фичей, shape [n_samples,2]
    :returns: расширенная матрица фичей, shape [n_samples,6]
    """

    # your code here
    # ヽ(♡‿♡)ノ
    

__(0.5 балла) Задание 2.__ Для классификации будем использовать логистическую регрессию. 

$$ a(x; w) = \langle w, x \rangle $$
$$ P( y=1 \; \big| \; x, \, w) = \dfrac{1}{1 + \exp(- \langle w, x \rangle)} = \sigma(\langle w, x \rangle)$$


In [ ]:
def probability(X, w):
    """
    Принимает на вход матрицу фичей и вектор весов
    Возвращает предсказание вероятность того, что y = 1 при фиксированных x, P(y=1|x)
    
    :param X: расширенная матрица фичей [n_samples,6] (expanded)
    :param w: вектор весов [6]
    :returns: вектор вероятностей
    """

    # your code here
    # ฅ^•ﻌ•^ฅ 

__(0.5 балла) Задание 3.__ Для логистической регрессии оптимальный параметр находится минимизацией кросс-энтропии: 

$$ L(w) =  - {1 \over \ell} \sum_{i=1}^\ell \left[ {y_i \cdot log P(y_i = 1 \, | \, x_i,w) + (1-y_i) \cdot log (1-P(y_i = 1 \, | \, x_i,w))}\right] $$



In [ ]:
def compute_loss(X, y, w):
    """
    Принимает на вход матрицу весов, вектор ответов и вектор весов.
    Выдаёт на выход значение функции потерь, расчитанное по формуле выше.
    """

    # your code here
    # (♡-_-♡) 

__(1.5 балла) Задание 4.__ Мы будем обучать модель методом градиентного спуска. Для этого нам придётся вычислить градиент функции потерь, представленной выше. Возьмите листочек, ручку и в бой! 

$$ \nabla_w L = ...$$

In [ ]:
def compute_grad(X, y, w):
    """
    Нахоит значение градиента.
    """

    # your code here
    # (´｡• ᵕ •｡`) 

Функция ниже предназначена для визуализации процесса обучения. 

In [ ]:
h = 0.01
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

def visualize(X, y, w, history):
    """С помощью магии matplolib выдаёт красоты результатов классификации"""
    Z = probability(expand(np.c_[xx.ravel(), yy.ravel()]), w)
    Z = Z.reshape(xx.shape)
    plt.subplot(1, 2, 1)
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    
    plt.subplot(1, 2, 2)
    plt.plot(history)
    plt.grid()
    ymin, ymax = plt.ylim()
    plt.ylim(0, ymax)
    display.clear_output(wait=True)
    plt.show()

In [ ]:
dummy_weights = np.linspace(-1, 1, 6)
visualize(X, y, dummy_weights, [0.5, 0.5, 0.25])

Пришло время обучить нашу модель. Для этого вам придётся дописать кусочки функций ниже. Обязательно попробуйте поменять гиперпараметры (размер батча и скорость обучения) и посмотреть как будет изменяться анимация. 

__(1.5 балла) Задание 5: Mini-batch SGD__

Берём несколько рандомных наблюдений и ищем градиент по ним! 

$$ w_t = w_{t-1} - \eta \dfrac{1}{m} \sum_{j=1}^m \nabla_w L(w_t, x_{i_j}, y_{i_j}) $$

In [ ]:
np.random.seed(42)
w = np.array([0, 0, 0, 0, 0, 1])

eta = 0.05 
alpha = 0.9 
nu = np.zeros_like(w)

n_iter = 100
batch_size = 4
loss = np.zeros(n_iter)
plt.figure(figsize=(12, 5))

for i in range(n_iter):
    
    # your code here
    # ヾ(๑╹◡╹)ﾉ

visualize(X, y, w, loss)
plt.clf()

__(0.5 балла) Задание 6: Momentum SGD__

Momentum это метод, который помогает стохастическому градиентному спуску сохранять направление движения. Это осуществляется за счёт добавления в выражение дополнительного слагаемого: накопленного за предыдущие шаги градиента с весом $\alpha$. 
<br>
<br>

$$ \nu_t = \alpha \nu_{t-1} + \eta\dfrac{1}{m} \sum_{j=1}^m \nabla_w L(w_t, x_{i_j}, y_{i_j}) $$
$$ w_t = w_{t-1} - \nu_t$$


In [ ]:
np.random.seed(42)
w = np.array([0, 0, 0, 0, 0, 1])

eta = 0.05 
alpha = 0.9 
nu = np.zeros_like(w)

n_iter = 100
batch_size = 4
loss = np.zeros(n_iter)
plt.figure(figsize=(12, 5))

for i in range(n_iter):
    
    # your code here
    # ٩(ˊ〇ˋ*)و

visualize(X, y, w, loss)
plt.clf()

__(0.5 балла) Задание 7: RMSprop__

В этом блоке реализуем RMSprop. Эта вариация градиентного спуска позволяет изменять скорость обучения индивидуально для каждого параметра. 

$$ G_t^j = \alpha G_{t-1}^j + (1 - \alpha) g_{tj}^2 $$
$$ w_t^j = w_{t-1}^j - \dfrac{\eta}{\sqrt{G_t^j + \varepsilon}} g_{tj} $$

In [ ]:
np.random.seed(42)

w = np.array([0, 0, 0, 0, 0, 1.])

eta = 0.1 
alpha = 0.9 
g2 = np.zeros_like(w)
eps = 1e-8

n_iter = 100
batch_size = 4
loss = np.zeros(n_iter)
plt.figure(figsize=(12,5))
for i in range(n_iter):

    # your code here
    # (っ˘ڡ˘ς) 

visualize(X, y, w, loss)
plt.clf()

__(0.5 балла) Задание 8: Adam.__ Реализуйте Adam.

In [ ]:
# your code here
# (⌒_⌒;)

__(1 балл, бонус) Задание 9\*.__ Выясните, что такое weight decay и реализуйте его для базовой версии SGD. Правда ли, что эта модернизация улучшает обучение?

In [ ]:
# your code here
# ( ￣ー￣)

# Часть 2: откуда в ridge штраф (4 балла)

У авиакомпании есть данные по 16 рейсам: дальность $x$ в тысячах километров и расход топлива $y$ в тоннах. 

Мы хотим обучить простейшую модель $a(x) = w x$ — прикидывать расход на новые маршруты. Для простоты будем считать, что $x$, и $y$ центрированы: из каждого вычтено среднее.

Функция потерь

$$
Q(w) = \sum_{i=1}^n (y_i - w x_i)^2,
$$

её безусловный минимум $\hat w = \dfrac{\sum x_i y_i}{\sum x_i^2}$. 

**Ridge-регрессия** — та же задача, но с ограничением на размер веса: $w^2 \le C$. Мы не даём модели тратить «слишком много веса» — ровно то, что делал weight decay в бонусе 9\*. В этой части мы увидим, что привычный штраф $\lambda w^2$ — это ровно множитель Лагранжа такого ограничения.

In [ ]:
distance = np.array([0.6, 1.6, 1.7, 1.8, 1.9, 1.9, 2.6, 2.7, 2.8, 3.0, 3.4, 4.0, 4.1, 4.2, 4.4, 4.5])
fuel     = np.array([2.9, 7.2, 6.2, 7.1, 7.0, 8.4, 10.0, 11.7, 11.9, 12.3, 11.6, 15.5, 16.3, 16.8, 16.0, 17.3])

x_r = distance - distance.mean()
y_r = fuel - fuel.mean()
S_xy = np.sum(x_r * y_r)
S_xx = np.sum(x_r ** 2)
w_ols = S_xy / S_xx
print(f"Σxy = {S_xy:.3f}, Σx² = {S_xx:.3f}, безусловный минимум ŵ = {w_ols:.4f}, ŵ² = {w_ols ** 2:.3f}")

def Q_r(w):
    return np.sum((y_r - w * x_r) ** 2)

plt.figure(figsize=(6, 4))
plt.scatter(x_r, y_r)
plt.plot(x_r, w_ols * x_r, color="gray", lw=1, label=f"ŵ = {w_ols:.2f}")
plt.xlabel("дальность (центрированная)"); plt.ylabel("топливо (центрированное)"); plt.legend()
plt.show()

__(2 балла) Задание 10: лагранжиан ridge__

Задача: $Q(w) \to \min_w$ при $w^2 \le C$.

1. Выпишите лагранжиан $\mathcal L(w, \lambda) = Q(w) + \lambda (w^2 - C)$ и решите $\partial \mathcal L / \partial w = 0$ относительно $w$. Получится формула $w_\lambda$, зависящая от $\lambda$. Сравните её с $\hat w$: что изменилось?
2. Раскройте скобки в $\mathcal L$: часть, зависящая от $w$, — это $Q(w) + \lambda w^2$. Это и есть функция потерь ridge-регрессии. Штраф $\lambda w^2$ — не придуманная добавка, а множитель Лагранжа ограничения $w^2 \le C$.
3. Что происходит с $w_\lambda$ при $\lambda \to 0$? При $\lambda \to \infty$?

Все вычисления приложите к файлу в виде фотки или markdown. Получившуюся формулу для веса вбейте ниже. 

In [ ]:
def w_ridge(lam):
    # YOUR CODE HERE: w_λ из задания 10
    # ∑(°ロ°)
    return ...

__(2 балла) Задание 11: связь между $C$ и $\lambda$__

Ограничение $w^2 \le C$ и штраф $\lambda w^2$ — две записи одной задачи, но в одной параметр $C$, в другой $\lambda$. Свяжите их, разобрав два случая.

1. Если $\hat w^2 \le C$, ограничение не мешает: $\lambda = 0$ и $w = \hat w$.
2. Если $\hat w^2 > C$, ограничение активно: $w_\lambda^2 = C$. Подставьте формулу $w_\lambda$ из задания 10 и выразите $\lambda$ через $C$ (считайте $S_{xy} > 0$, тогда $w_\lambda = +\sqrt C$).

Впишите в код формулу $w_\lambda$ из задания 10 и $\lambda(C)$. Затем для $C = 4$ нарисуйте параболу $Q(w)$, закрасьте допустимый отрезок $[-\sqrt C, \sqrt C]$ и отметьте безусловный минимум $\hat w$ и условный $w_{\lambda(C)}$ — где он лежит относительно границы?

Все вычисления приложите к файлу в виде фотки или markdown. Ниже вбейте получившиеся формулы.

In [ ]:
def lam_of_C(C):
    # YOUR CODE HERE: λ(C) с учётом обоих случаев
    # (☉_☉)
    return ...

C = 4.0
lam_C = lam_of_C(C)

# YOUR CODE HERE: картинка — парабола Q(w), допустимый отрезок, две точки

In [ ]:
check(abs(w_ridge(0) - w_ols) < 1e-12, "при λ = 0 ridge совпадает с обычной регрессией")
check(w_ridge(10) < w_ridge(1) < w_ridge(0), "с ростом λ вес сжимается")
for lam in [0.5, 3.0, 20.0]:
    d = numeric_derivative(lambda w: Q_r(w) + lam * w ** 2, w_ridge(lam))
    check(abs(d) < 1e-5, f"λ = {lam}: производная Q + λw² в точке w_λ равна нулю — формула из задания 10 верна")
for C_test in [1.0, 4.0, 9.0]:
    check(abs(w_ridge(lam_of_C(C_test)) ** 2 - C_test) < 1e-9, f"C = {C_test}: ridge с λ(C) = {lam_of_C(C_test):.3f} даёт ровно w² = C")
check(lam_of_C(20.0) == 0.0, f"C = 20 > ŵ² = {w_ols ** 2:.2f}: ограничение не активно, λ = 0")
check(Q_r(w_ridge(lam_C)) > Q_r(w_ols), "условный минимум хуже безусловного на обучении: за ограничение платим качеством")

**Вопросы (по 0.5):**

- Множитель Лагранжа — это всегда «цена» ограничения: на сколько улучшится оптимум, если ограничение ослабить на единицу. Ценой чего является $\lambda$ в ridge и в каких единицах он измеряется?

**Ответ:**

- Обычно ridge записывают не через $C$, а сразу через $\lambda$, и подбирают $\lambda$ по отложенной выборке. Почему так удобнее, если две постановки эквивалентны? Подсказка: посмотрите, от чего зависит $\lambda(C)$.

**Ответ:**


## Что почитать

* Sebastian Ruder, [An overview of gradient descent optimization algorithms](https://www.ruder.io/optimizing-gradient-descent/) — все пять методов из части 1 в одном месте, с формулами и картинками.
* Николенко, Кадурин, Архангельская, *Глубокое обучение. Погружение в мир нейронных сетей*, глава 4 — то же по-русски и с историей, откуда что взялось.
* Boyd, Vandenberghe, *Convex Optimization*, глава 5 — двойственность и множители Лагранжа; раздел 5.6 про то, почему множитель — это чувствительность оптимума к ограничению.
* [Задачник по глубокому обучению, листок про градиентный спуск](https://fulyankin.github.io/deep_learning_masha_book/problem_set_02_gradient/intro_02.html).